# Mô tả

## Mục tiêu
Trong notebook này, nhóm sẽ thực hiện:
1.  **Xây dựng giả thuyết:** Đề xuất 2 cách tiếp cận khác nhau để trả lời câu hỏi ML: **dự đoán `salary_in_usd`**.
2.  **Feature Engineering:** Biến đổi dữ liệu thô thành các features phù hợp cho từng giả thuyết.
3.  **Data Splitting:** Chia tập dữ liệu thành Train/Test và lưu trữ để chuẩn bị cho bước Modeling.

## Input & Output
- **Input:** `../data/original_data.csv` (Dữ liệu gốc).
- **Output:** - `../data/train_hypo_1.csv`, `../data/test_hypo_1.csv`
    - `../data/train_hypo_2.csv`, `../data/test_hypo_2.csv`

## Import thư viện

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split


# Cấu hình hiển thị để dễ quan sát dữ liệu
pd.set_option('display.max_columns', None)

# Đường dẫn file dữ liệu (Cập nhật nếu cần)
FILE_PATH = '../data/original_data.csv'
OUTPUT_DIR = '../data/'

# Đọc dữ liệu
try:
    df = pd.read_csv(FILE_PATH)
    print(f"Đã đọc dữ liệu thành công. Kích thước: {df.shape}")
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file tại {FILE_PATH}")


Đã đọc dữ liệu thành công. Kích thước: (105434, 11)


# 1. Giả thuyết 1: Thâm niên và vị trí địa lý

## 1.1. Cơ sở lựa chọn
Dựa trên quá trình EDA và quan sát thực tế thị trường lao động, nhóm nhận thấy:
1.  **Kinh nghiệm (Experience Level):** Có tương quan dương rất mạnh với mức lương. Đường nối các điểm lương trung vị của từng mức kinh nghiệm là một đường thẳng tuyến tính đồng biến.
2.  **Vị trí công ty (Company Location):** Mỹ (US) là thị trường chi trả lương cao nhất thế giới cho ngành dữ liệu. Sự chênh lệch giữa công ty tại Mỹ và các quốc gia khác là rất lớn, lấn át các yếu tố địa lý nhỏ lẻ khác.
3.  **Lạm phát theo năm (Work Year):** Lương có xu hướng tăng nhẹ theo từng năm do lạm phát và nhu cầu thị trường.

## 1.2. Mục tiêu thực nghiệm
Mục tiêu của giả thuyết này là xây dựng một **Baseline Model**. 

Nhóm muốn kiểm chứng xem: Liệu chỉ với 3 yếu tố cốt lõi là:
- **"Bạn có nhiều kinh nghiệm thế nào?"** (Experience)
- **"Công ty bạn ở đâu?"** (US/Non-US)
- **"Thời điểm nào?"** (Year) 

Đã đủ để giải thích phần lớn sự biến thiên của mức lương hay chưa? 

Nếu mô hình này có kết quả tốt, chứng tỏ thị trường định giá nhân sự chủ yếu dựa trên thâm niên và địa lý.

## 1.3. Phương pháp thực hiện

Để đưa các dữ liệu trên vào mô hình hồi quy, nhóm thực hiện Feature Engineering như sau:

* **Experience Level:** Sử dụng **Ordinal Encoding** (Mã hóa thứ tự) vì cấp bậc có tính chất tăng dần: EN (0) < MI (1) < SE (2) < EX (3).
* **Company Location:** Sử dụng **Binary Encoding** (Mã hóa nhị phân). Gom nhóm tất cả các quốc gia thành 2 nhóm: `1` (nếu là US) và `0` (nếu không phải US). Điều này giúp giảm chiều dữ liệu đáng kể so với One-hot encoding hàng chục quốc gia.
* **Work Year:** Giữ nguyên dạng số thực để mô hình bắt được xu hướng tuyến tính theo thời gian.

## 1.4. Hàm giả thuyết

Với mô hình Linear Regression, phương trình dự kiến sẽ có dạng:

$$h_\theta(x) = \theta_0 + \theta_1 * (\text{experience\_level}) + \theta_2 * (\text{is\_US\_company}) + \theta_3 * (\text{work\_year})$$

Trong đó:
* $h_\theta(x)$: Mức lương dự đoán (`salary_in_usd`).
* $\theta_0$: Bias.
* $\theta_1, \theta_2, \theta_3$: Trọng số hồi quy cho từng đặc trưng. Nhóm kỳ vọng các hệ số này đều mang dấu dương (+).

## 1.5. Code

### Helper Functions

In [3]:
# --- Helper Functions cho H1 ---

def map_experience_level(level):
    """
    Chuyển đổi mức độ kinh nghiệm từ chuỗi ký tự sang số nguyên (Ordinal Encoding).
    
    Input:
        level (str): Mã mức độ kinh nghiệm ('EN', 'MI', 'SE', 'EX').
    
    Output:
        int: Giá trị số tương ứng (0, 1, 2, 3). Trả về 0 nếu không khớp.
    """
    mapping = {
        'EN': 0,  # Entry-level
        'MI': 1,  # Mid-level
        'SE': 2,  # Senior-level
        'EX': 3   # Executive-level
    }
    return mapping.get(level, 0)

def create_hypothesis_1_data(df_input):
    """
    Xử lý dữ liệu và trích xuất features cho Giả thuyết 1.
    
    Input:
        df_input (pd.DataFrame): DataFrame dữ liệu gốc.
        
    Output:
        pd.DataFrame: DataFrame chỉ chứa các features đã xử lý cho H1 và cột target.
    """
    df_temp = df_input.copy()
    
    # 1. Ordinal Encoding cho Experience Level
    # Lý do: Biến đổi dữ liệu hạng mục có thứ tự sang dạng số để mô hình hồi quy hiểu được quan hệ lớn/bé.
    df_temp['experience_level_encoded'] = df_temp['experience_level'].apply(map_experience_level)
    
    # 2. Binary Encoding cho Company Location
    # Lý do: Tập trung vào sự khác biệt lớn nhất (Mỹ vs Phần còn lại) để tránh nhiễu từ các nước có ít mẫu.
    df_temp['is_US_company'] = df_temp['company_location'].apply(lambda x: 1 if x == 'US' else 0)
    
    # 3. Lựa chọn cột (Feature Selection)
    features = ['work_year', 'experience_level_encoded', 'is_US_company', 'salary_in_usd']
    
    return df_temp[features]

### Thực thi

In [4]:
# --- Thực thi ---
print("--- Đang xử lý Giả thuyết 1 ---")
df_h1 = create_hypothesis_1_data(df)

# Hiển thị 5 dòng đầu để kiểm tra
display(df_h1.head())
print(f"Kích thước tập dữ liệu H1: {df_h1.shape}")

--- Đang xử lý Giả thuyết 1 ---


,work_year,experience_level_encoded,is_US_company,salary_in_usd
0,2025,0,0,69120
1,2025,0,0,50160
2,2025,0,1,158113
3,2025,0,1,87795
4,2025,3,1,351410


Kích thước tập dữ liệu H1: (105434, 4)


### Chia tập train, test

In [5]:
# Tách features (X) và target (y)
X_h1 = df_h1.drop(columns=['salary_in_usd'])
y_h1 = df_h1['salary_in_usd']

# Chia tập Train/Test theo tỷ lệ 80/20
# random_state=42 giúp cố định kết quả chia ngẫu nhiên để tái lập thí nghiệm sau này
X_train_h1, X_test_h1, y_train_h1, y_test_h1 = train_test_split(X_h1, y_h1, test_size=0.2, random_state=42)

# Ghép lại thành DataFrame để lưu file tiện lợi
train_h1 = pd.concat([X_train_h1, y_train_h1], axis=1)
test_h1 = pd.concat([X_test_h1, y_test_h1], axis=1)

# Lưu ra file CSV
train_path_h1 = os.path.join(OUTPUT_DIR, 'train_hypo_1.csv')
test_path_h1 = os.path.join(OUTPUT_DIR, 'test_hypo_1.csv')

train_h1.to_csv(train_path_h1, index=False)
test_h1.to_csv(test_path_h1, index=False)

print(f"Đã lưu file Train H1 tại: {train_path_h1} | Shape: {train_h1.shape}")
print(f"Đã lưu file Test H1 tại: {test_path_h1}   | Shape: {test_h1.shape}")

Đã lưu file Train H1 tại: ../data/train_hypo_1.csv | Shape: (84347, 4)
Đã lưu file Test H1 tại: ../data/test_hypo_1.csv   | Shape: (21087, 4)


# 2. Giả thuyết 2: Chức vụ công việc (Job Role) và Quy mô công ty

## 2.1. Cơ sở lựa chọn
Giả thuyết 1 có thể quá đơn giản hóa. Thực tế, lương còn phụ thuộc vào các yếu tố chi tiết hơn:
1.  **Chức danh (Job Role):** Một "Data Scientist" thường có lương khác với "Data Analyst" hay "Data Manager", dù cùng số năm kinh nghiệm.
2.  **Quy mô công ty (Company Size):** Các công ty lớn (Large) thường có nguồn lực tài chính mạnh hơn các công ty nhỏ (Small), dẫn đến khung lương khác nhau.
3.  **Làm việc từ xa (Remote Ratio):** Hình thức làm việc (Remote/On-site) cũng là một yếu tố phản ánh văn hóa công ty và có thể ảnh hưởng đến lương (ví dụ: remote cho công ty nước ngoài).

## 2.2. Mục tiêu thực nghiệm
Mục tiêu là kiểm tra xem **"Độ phức tạp có mang lại hiệu quả?"**.

Nhóm muốn quan sát xem việc thêm các biến chi tiết (Role, Size, Remote) có giúp giảm sai số dự báo (RMSE) và tăng độ chính xác ($R^2$) đáng kể so với Baseline Model ở H1 hay không. Đồng thời, nhóm muốn xem mô hình đánh giá vai trò nào (Scientist, Engineer, hay Manager) đóng góp tích cực nhất vào mức lương.

## 2.3. Phương pháp thực hiện

* **Job Title:** Vì có quá nhiều chức danh (gây nhiễu), nhóm sẽ **Gom nhóm (Grouping)** các chức danh tương đồng (ví dụ: 'Head of Data', 'Data Lead' -> 'Manager_Lead'), sau đó sử dụng **One-Hot Encoding**.
* **Company Size:** Sử dụng **Ordinal Encoding** (S < M < L) vì quy mô có tính thứ tự.
* **Remote Ratio:** Giữ nguyên giá trị số (0, 50, 100) hoặc coi như biến định lượng.
* **Experience Level & Work Year:** Giữ nguyên cách xử lý như H1 vì đây là các biến nền tảng quan trọng.

## 2.4. Hàm giả thuyết

Với giả thuyết này, phương trình hồi quy tuyến tính mở rộng sẽ là:

$$h_\theta(x) = \theta_0 + \theta_1(Exp) + \theta_2(Size) + \theta_3(Remote) + \sum_{i=1}^{k} \theta_{4,i} * (JobRole\_i)$$

Trong đó:
* $\theta_1(Exp)$: Hệ số cho kinh nghiệm.
* $\theta_2(Size)$: Hệ số cho quy mô công ty.
* $\sum \theta_{4,i} * (JobRole\_i)$: Tổng các hệ số tương ứng với các nhóm nghề nghiệp sau khi đã One-hot encoding (ví dụ: $\theta_{Scientist} * Is\_Scientist$).

## 2.5. Code

### Helper Functions

In [7]:
# --- Helper Functions cho H2 ---

def map_company_size(size):
    """
    Chuyển đổi quy mô công ty sang giá trị số (Ordinal Encoding).
    
    Input: size (str) - 'S', 'M', 'L'
    Output: int - 0, 1, 2
    """
    mapping = {'S': 0, 'M': 1, 'L': 2}
    return mapping.get(size, 1) # Mặc định là M nếu thiếu dữ liệu

def categorize_job_title(title):
    """
    Gom nhóm các Job Title chi tiết thành các nhóm lớn (Reducing Cardinality).
    
    Input: title (str) - Tên chức danh gốc
    Output: str - Tên nhóm chức danh mới
    """
    title = str(title).lower()
    
    # Ưu tiên tìm kiếm theo từ khóa
    if any(x in title for x in ['manager', 'lead', 'principal', 'head', 'director']):
        return 'Manager_Lead'
    elif any(x in title for x in ['scientist', 'science', 'research']):
        return 'Data_Scientist'
    elif any(x in title for x in ['engineer', 'architect']):
        return 'Data_Engineer'
    elif any(x in title for x in ['analyst', 'analytics']):
        return 'Data_Analyst'
    elif any(x in title for x in ['machine learning', 'ml', 'ai', 'vision', 'nlp']):
        return 'ML_AI_Engineer'
    else:
        return 'Other'

def create_hypothesis_2_data(df_input):
    """
    Xử lý dữ liệu và trích xuất features cho Giả thuyết 2.
    """
    df_temp = df_input.copy()
    
    # 1. Ordinal Encoding: Exp & Company Size
    df_temp['experience_level_encoded'] = df_temp['experience_level'].apply(map_experience_level)
    df_temp['company_size_encoded'] = df_temp['company_size'].apply(map_company_size)
    
    # 2. Grouping Job Title
    df_temp['job_group'] = df_temp['job_title'].apply(categorize_job_title)
    
    # 3. One-Hot Encoding cho Job Group
    # drop_first=True: Loại bỏ 1 cột để tránh hiện tượng đa cộng tuyến hoàn hảo (Dummy Variable Trap)
    job_dummies = pd.get_dummies(df_temp['job_group'], prefix='job', drop_first=True)
    
    # Ghép các cột dummy vào dataframe tạm
    df_temp = pd.concat([df_temp, job_dummies], axis=1)
    
    # 4. Feature Selection
    # Lấy các features cơ bản + các cột dummy vừa tạo
    feature_cols = ['work_year', 'experience_level_encoded', 'company_size_encoded', 'remote_ratio']
    feature_cols.extend(job_dummies.columns.tolist()) # Thêm danh sách các cột job_...
    feature_cols.append('salary_in_usd') # Target
    
    return df_temp[feature_cols]

### Thực thi

In [8]:
# --- Thực thi ---
print("--- Đang xử lý Giả thuyết 2 ---")
df_h2 = create_hypothesis_2_data(df)

display(df_h2.head())
print(f"Kích thước tập dữ liệu H2: {df_h2.shape}")
print(f"Danh sách Features H2: {list(df_h2.columns)}")

--- Đang xử lý Giả thuyết 2 ---


,work_year,experience_level_encoded,company_size_encoded,remote_ratio,job_Data_Engineer,job_Data_Scientist,job_ML_AI_Engineer,job_Manager_Lead,job_Other,salary_in_usd
0,2025,0,1,0,False,False,False,False,True,69120
1,2025,0,1,0,False,False,False,False,True,50160
2,2025,0,1,0,True,False,False,False,False,158113
3,2025,0,1,0,True,False,False,False,False,87795
4,2025,3,1,0,True,False,False,False,False,351410


Kích thước tập dữ liệu H2: (105434, 10)
Danh sách Features H2: ['work_year', 'experience_level_encoded', 'company_size_encoded', 'remote_ratio', 'job_Data_Engineer', 'job_Data_Scientist', 'job_ML_AI_Engineer', 'job_Manager_Lead', 'job_Other', 'salary_in_usd']


### Chia tập train, test

In [9]:
# Tách features và target
X_h2 = df_h2.drop(columns=['salary_in_usd'])
y_h2 = df_h2['salary_in_usd']

# Chia tập Train/Test (Vẫn giữ random_state=42 để đồng bộ với H1 về mặt lấy mẫu)
X_train_h2, X_test_h2, y_train_h2, y_test_h2 = train_test_split(X_h2, y_h2, test_size=0.2, random_state=42)

# Ghép lại
train_h2 = pd.concat([X_train_h2, y_train_h2], axis=1)
test_h2 = pd.concat([X_test_h2, y_test_h2], axis=1)

# Lưu file CSV
train_path_h2 = os.path.join(OUTPUT_DIR, 'train_hypo_2.csv')
test_path_h2 = os.path.join(OUTPUT_DIR, 'test_hypo_2.csv')

train_h2.to_csv(train_path_h2, index=False)
test_h2.to_csv(test_path_h2, index=False)

print(f"Đã lưu file Train H2 tại: {train_path_h2} | Shape: {train_h2.shape}")
print(f"Đã lưu file Test H2 tại: {test_path_h2}   | Shape: {test_h2.shape}")

Đã lưu file Train H2 tại: ../data/train_hypo_2.csv | Shape: (84347, 10)
Đã lưu file Test H2 tại: ../data/test_hypo_2.csv   | Shape: (21087, 10)
